# Multivariate Workflow Solution: Bayesian VAR Analysis

**Solution notebook** for the exercise in `02_multivariate_workflow.ipynb`.

This notebook extends the multivariate analysis with a **Bayesian VAR (BVAR)** model,
comparing it against the standard VAR and VECM approaches.

## Pipeline

1. Data loading and exploratory analysis
2. Unit root tests for each variable
3. VAR lag selection and Johansen cointegration
4. **Decision tree: VAR vs VECM vs ARDL (resolved)**
5. Model estimation: VAR, VECM, **BVAR (Minnesota)**, **BVAR (Normal-Wishart)**
6. IRF and FEVD (VAR + BVAR comparison)
7. Granger causality
8. Structural VAR (SVAR Cholesky)
9. Diebold-Yilmaz spillover index
10. Multivariate forecasting (VAR vs BVAR comparison)
11. Report generation

**Dataset**: `us_macro_quarterly.csv` - GDP growth, inflation, fed funds, unemployment (1975-2024)

## Pipeline Flowchart (with BVAR)

```
US Macro Data (4 variables, 200 obs)
   |
   v
[1. Exploratory Analysis] ---> Plots, correlations, descriptive stats
   |
   v
[2. Unit Root Tests] -------> ADF + KPSS for each variable
   |                            => Determine I(0)/I(1) for each
   v
[3. Lag Selection] ----------> AIC/BIC/HQ across p=1..8
   |
   v
[4. Cointegration] ----------> Johansen trace + max-eigenvalue
   |
   v
[5. DECISION TREE] =========> All I(0)? -------> VAR levels
   |                           All I(1)+coint? -> VECM
   |                           All I(1) no coint -> VAR diff
   |                           Mixed? -----------> ARDL
   v
[6. Estimation] ============>  VAR(p)
   |                           VECM(p, rank=r)
   |                           BVAR(p, minnesota)
   |                           BVAR(p, normal_wishart)
   v
[7. IRF + FEVD] -----------> Compare VAR vs BVAR IRFs
   v
[8. Granger Causality] -----> Pairwise F-tests
   v
[9. SVAR] -----------------> Cholesky identification
   v
[10. Spillover] ------------> Diebold-Yilmaz index
   v
[11. Forecasting] ----------> VAR vs BVAR with posterior intervals
   v
[12. Report] ----------------> generate_comparison_report()
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import warnings
warnings.filterwarnings('ignore')

# chronobox imports
from chronobox import VAR, VECM, SVAR, BayesianVAR
from chronobox.models import ARDL
from chronobox.analysis import SpilloverIndex, granger_causality
from chronobox.tests_stat import adf_test, kpss_test, ljung_box_test
from chronobox.visualization import (
    plot_irf, plot_fevd, plot_heatmap, plot_network, set_theme
)

import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'utils'))
from report_generator import generate_comparison_report

set_theme('professional')

data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
output_dir = os.path.join(os.path.dirname(os.getcwd()), 'outputs')
os.makedirs(output_dir, exist_ok=True)

print('chronobox imported successfully')

---
## 1. Data Loading and Exploratory Analysis

In [2]:
df = pd.read_csv(os.path.join(data_dir, 'us_macro_quarterly.csv'), parse_dates=['date'])
df = df.set_index('date')
df.index.freq = 'QS'

var_names = ['gdp', 'inflation', 'fed_funds', 'unemployment']

print(f'Period: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Observations: {len(df)}')
print(f'Variables: {var_names}')
print()
print(df.describe().round(4).to_string())

In [3]:
data = df[var_names].values
T, K = data.shape
print(f'Data matrix: T={T}, K={K}')

In [4]:
# GRAPH 1: Plot all variables
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
titles = ['GDP Growth (%)', 'Inflation (%)', 'Federal Funds Rate (%)', 'Unemployment (%)']
colors = ['steelblue', 'darkorange', 'green', 'red']

for ax, col, title, color in zip(axes.flat, var_names, titles, colors):
    ax.plot(df.index, df[col], color=color, linewidth=1.2)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=df[col].mean(), color='gray', linestyle='--', alpha=0.5)

plt.suptitle('US Macroeconomic Variables (Quarterly)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [5]:
# GRAPH 2: Correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df[var_names].corr()
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_xticks(range(len(var_names)))
axes[0].set_yticks(range(len(var_names)))
axes[0].set_xticklabels(var_names, rotation=45, ha='right')
axes[0].set_yticklabels(var_names)
for i in range(len(var_names)):
    for j in range(len(var_names)):
        axes[0].text(j, i, f'{corr.iloc[i, j]:.2f}',
                     ha='center', va='center', fontsize=10,
                     color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title('Correlation Matrix', fontsize=12)

axes[1].scatter(df['unemployment'], df['gdp'], alpha=0.5, c='steelblue', s=20)
z = np.polyfit(df['unemployment'], df['gdp'], 1)
p_line = np.poly1d(z)
x_line = np.linspace(df['unemployment'].min(), df['unemployment'].max(), 100)
axes[1].plot(x_line, p_line(x_line), 'r--', linewidth=1.5)
axes[1].set_xlabel('Unemployment (%)')
axes[1].set_ylabel('GDP Growth (%)')
axes[1].set_title('GDP vs Unemployment', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Unit Root Tests

| ADF | KPSS | Conclusion |
|-----|------|------------|
| Reject | Fail to reject | I(0) - stationary |
| Fail to reject | Reject | I(1) - unit root |

In [6]:
print('=' * 70)
print(f'{"Variable":<15} {"ADF stat":>10} {"ADF p":>8} {"KPSS stat":>10} {"KPSS p":>8} {"Order":>6}')
print('=' * 70)

integration_orders = {}
test_details = {}

for col in var_names:
    y_col = df[col].values
    adf = adf_test(y_col, regression='c')
    kpss = kpss_test(y_col, regression='c')

    if adf.reject_at_5pct and not kpss.reject_at_5pct:
        order = 'I(0)'
    else:
        dy = np.diff(y_col)
        adf_d = adf_test(dy, regression='c')
        order = 'I(1)' if adf_d.reject_at_5pct else 'I(2)?'

    integration_orders[col] = order
    test_details[col] = {
        'adf_level': {
            'statistic': float(adf.statistic),
            'pvalue': float(adf.pvalue),
            'reject': bool(adf.reject_at_5pct)
        },
        'kpss_level': {
            'statistic': float(kpss.statistic),
            'pvalue': float(kpss.pvalue),
            'reject': bool(kpss.reject_at_5pct)
        },
        'integration_order': order
    }

    print(f'{col:<15} {adf.statistic:>10.4f} {adf.pvalue:>8.4f} '
          f'{kpss.statistic:>10.4f} {kpss.pvalue:>8.4f} {order:>6}')

print('=' * 70)
print(f'\nIntegration orders: {integration_orders}')

In [7]:
# Save tests
multivariate_tests = {
    'dataset': 'us_macro_quarterly.csv',
    'n_obs': T,
    'n_vars': K,
    'variables': var_names,
    'integration_orders': integration_orders,
    'test_details': test_details
}

with open(os.path.join(output_dir, 'multivariate_tests.json'), 'w') as f:
    json.dump(multivariate_tests, f, indent=2)

print(f'Saved: {output_dir}/multivariate_tests.json')

---
## 3. VAR Lag Selection and Johansen Cointegration

In [8]:
# Lag selection
print('=== VAR Lag Selection ===')
print(f'{"Lags":>5} {"AIC":>12} {"BIC":>12} {"HQ":>12}')
print('-' * 45)

ic_results = {}
for p in range(1, 9):
    var_temp = VAR(lags=p, trend='c')
    res_temp = var_temp.fit(data, names=var_names)
    ic_results[p] = {'aic': float(res_temp.aic), 'bic': float(res_temp.bic),
                     'hqic': float(res_temp.hqic)}
    print(f'{p:>5} {res_temp.aic:>12.4f} {res_temp.bic:>12.4f} {res_temp.hqic:>12.4f}')

best_aic = min(ic_results, key=lambda x: ic_results[x]['aic'])
best_bic = min(ic_results, key=lambda x: ic_results[x]['bic'])
best_hq = min(ic_results, key=lambda x: ic_results[x]['hqic'])

print(f'\nOptimal: AIC={best_aic}, BIC={best_bic}, HQ={best_hq}')
p_opt = best_aic
print(f'Selected: p = {p_opt} (AIC)')

In [9]:
# Johansen cointegration
vecm_test = VECM(lags=p_opt, deterministic='ci')
johansen = vecm_test.johansen_test(data)

print('=== Johansen Cointegration Test ===')
print(f'\nTrace Test:')
print(f'{"H0: rank <= r":>15} {"Trace Stat":>12} {"95% CV":>10} {"Decision":>15}')
print('-' * 55)
for r in range(K):
    stat = johansen.trace_stat[r]
    cv = johansen.trace_crit[r, 1]
    reject = stat > cv
    print(f'{"r <= " + str(r):>15} {stat:>12.4f} {cv:>10.4f} '
          f'{"Reject" if reject else "Fail to reject":>15}')

print(f'\nMax Eigenvalue Test:')
print(f'{"H0: rank = r":>15} {"Max-Eig Stat":>12} {"95% CV":>10} {"Decision":>15}')
print('-' * 55)
for r in range(K):
    stat = johansen.max_eig_stat[r]
    cv = johansen.max_eig_crit[r, 1]
    reject = stat > cv
    print(f'{"r = " + str(r):>15} {stat:>12.4f} {cv:>10.4f} '
          f'{"Reject" if reject else "Fail to reject":>15}')

coint_rank = int(johansen.rank_trace)
print(f'\nCointegration rank (trace): {coint_rank}')

---
## 4. Decision Tree: Model Selection (Resolved)

```
Are all variables I(0)?
  |-- Yes --> VAR in levels
  |-- No  --> Are all variables I(1)?
                |-- No  --> Mixed orders --> ARDL bounds test
                |-- Yes --> Cointegrated?
                              |-- Yes --> VECM (rank = r)
                              |-- No  --> VAR in first differences
```

In [10]:
# Resolve decision tree
orders = list(integration_orders.values())
n_i0 = sum(1 for o in orders if o == 'I(0)')
n_i1 = sum(1 for o in orders if o == 'I(1)')

print('=== Model Selection Decision Tree ===')
print(f'Integration orders: {integration_orders}')
print(f'  I(0): {n_i0}, I(1): {n_i1}')

if n_i0 == K:
    decision = 'VAR_LEVELS'
    decision_text = 'All variables are I(0) => VAR in levels'
elif n_i1 == K:
    if coint_rank > 0:
        decision = 'VECM'
        decision_text = f'All I(1), {coint_rank} cointegrating relation(s) => VECM'
    else:
        decision = 'VAR_DIFF'
        decision_text = 'All I(1), no cointegration => VAR in first differences'
else:
    decision = 'ARDL'
    decision_text = 'Mixed integration orders => ARDL'

print(f'\n=> DECISION: {decision_text}')
print(f'\n(Estimating VAR, VECM, and BVAR for comparison)')

In [11]:
# Save model selection
model_selection = {
    'integration_orders': integration_orders,
    'n_i0': n_i0,
    'n_i1': n_i1,
    'cointegration_rank_trace': coint_rank,
    'decision': decision,
    'decision_text': decision_text,
    'optimal_lag_aic': best_aic,
    'optimal_lag_bic': best_bic,
    'selected_lag': p_opt,
    'ic_values': {str(k): v for k, v in ic_results.items()}
}

with open(os.path.join(output_dir, 'multivariate_model_selection.json'), 'w') as f:
    json.dump(model_selection, f, indent=2)

print(f'Saved: {output_dir}/multivariate_model_selection.json')

---
## 5. Model Estimation

### 5a. VAR

In [12]:
var_model = VAR(lags=p_opt, trend='c')
var_result = var_model.fit(data, names=var_names)

print('=== VAR Summary ===')
print(f'Lag order:  {var_result.k_ar}')
print(f'Stable:     {var_result.is_stable}')
print(f'AIC: {var_result.aic:.4f}')
print(f'BIC: {var_result.bic:.4f}')
print(f'Coefs shape: {var_result.coefs.shape}')

### 5b. VECM

In [13]:
vecm_rank = max(coint_rank, 1)
vecm_model = VECM(lags=p_opt, coint_rank=vecm_rank, deterministic='ci')
vecm_result = vecm_model.fit(data, names=var_names)

print(f'=== VECM (rank={vecm_rank}) ===')
print(f'Cointegration rank: {vecm_result.coint_rank}')

print(f'\nAlpha (adjustment):')
alpha_df = pd.DataFrame(vecm_result.alpha, index=var_names,
                        columns=[f'EC{i+1}' for i in range(vecm_rank)])
print(alpha_df.round(4).to_string())

print(f'\nBeta (cointegrating vectors):')
beta_df = pd.DataFrame(vecm_result.beta[:K], index=var_names,
                       columns=[f'CV{i+1}' for i in range(vecm_rank)])
print(beta_df.round(4).to_string())

### 5c. Bayesian VAR

The Minnesota prior (Litterman, 1986) shrinks coefficients toward a random walk.
Key features:
- Own lags shrunk less than cross-variable lags
- Higher lags shrunk more aggressively
- Prevents overfitting in high-dimensional VARs

In [14]:
# BVAR Minnesota prior
bvar_mn = BayesianVAR(lags=p_opt, prior='minnesota')
bvar_mn_result = bvar_mn.fit(data, n_draws=5000, burnin=1000, seed=42)

print('=== Bayesian VAR (Minnesota) ===')
print(f'Lags: {bvar_mn_result.lags}')
print(f'Draws: 5000, Burnin: 1000')
print(f'Log marginal likelihood: {bvar_mn_result.log_marginal_likelihood:.4f}')

print(f'\nPosterior mean coefs (lag 1):')
coef_df = pd.DataFrame(bvar_mn_result.coefs_mean[0], index=var_names, columns=var_names)
print(coef_df.round(4).to_string())

print(f'\nPosterior mean sigma:')
sigma_df = pd.DataFrame(bvar_mn_result.sigma_mean, index=var_names, columns=var_names)
print(sigma_df.round(4).to_string())

In [15]:
# BVAR Normal-Wishart prior
bvar_nw = BayesianVAR(lags=p_opt, prior='normal_wishart')
bvar_nw_result = bvar_nw.fit(data, n_draws=5000, burnin=1000, seed=42)

print('=== Bayesian VAR (Normal-Wishart) ===')
print(f'Log marginal likelihood: {bvar_nw_result.log_marginal_likelihood:.4f}')

print(f'\nPosterior mean coefs (lag 1):')
coef_nw = pd.DataFrame(bvar_nw_result.coefs_mean[0], index=var_names, columns=var_names)
print(coef_nw.round(4).to_string())

In [16]:
# Compare BVAR priors
print('=== BVAR Prior Comparison ===')
print(f'{"Metric":<30} {"Minnesota":>15} {"Normal-Wishart":>15}')
print('-' * 62)
print(f'{"Log marginal likelihood":<30} '
      f'{bvar_mn_result.log_marginal_likelihood:>15.4f} '
      f'{bvar_nw_result.log_marginal_likelihood:>15.4f}')

mn_norm = np.linalg.norm(bvar_mn_result.coefs_mean)
nw_norm = np.linalg.norm(bvar_nw_result.coefs_mean)
print(f'{"Coefficient norm":<30} {mn_norm:>15.4f} {nw_norm:>15.4f}')

mn_std = np.mean(np.std(bvar_mn_result.coefs_draws, axis=0))
nw_std = np.mean(np.std(bvar_nw_result.coefs_draws, axis=0))
print(f'{"Avg posterior std":<30} {mn_std:>15.4f} {nw_std:>15.4f}')

preferred = 'minnesota' if bvar_mn_result.log_marginal_likelihood > bvar_nw_result.log_marginal_likelihood else 'normal_wishart'
print(f'\n=> Preferred prior: {preferred}')

---
## 6. Impulse Response Functions (IRF) and FEVD

In [17]:
# VAR IRF
irf_result = var_result.irf(periods=20, method='cholesky', sigs=0.95, runs=500)

print('=== VAR IRF ===')
print(f'Shape: {irf_result.irfs.shape}')
print(f'\nImpact matrix (period 0):')
impact_df = pd.DataFrame(
    irf_result.irfs[0],
    index=[f'Resp: {n}' for n in var_names],
    columns=[f'Shock: {n}' for n in var_names]
)
print(impact_df.round(4).to_string())

In [18]:
# GRAPH 3: IRF plot
fig = plot_irf(
    irf_results=irf_result,
    var_names=var_names,
    title='VAR Impulse Response Functions (Cholesky)',
    figsize=(14, 12)
)
plt.tight_layout()
plt.show()

In [19]:
# BVAR IRF comparison
bvar_irf = bvar_mn_result.irf(periods=20, method='cholesky')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
periods_range = np.arange(21)
responses = [
    (0, 2, 'GDP <- Fed Funds'),
    (1, 2, 'Inflation <- Fed Funds'),
    (3, 0, 'Unemployment <- GDP'),
    (2, 1, 'Fed Funds <- Inflation')
]
colors_r = ['steelblue', 'darkorange', 'red', 'green']

for ax, (resp, shock, title), color in zip(axes.flat, responses, colors_r):
    ax.plot(periods_range, irf_result.irfs[:, resp, shock],
            color=color, linewidth=2, label='VAR')
    ax.plot(periods_range, bvar_irf[:, resp, shock],
            color=color, linewidth=2, linestyle='--', label='BVAR')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('IRF Comparison: VAR vs BVAR', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [20]:
# FEVD
fevd_result = var_result.fevd(periods=20, method='cholesky')

print('=== FEVD at horizon 10 ===')
fevd_h10 = pd.DataFrame(
    fevd_result.decomp[10] * 100,
    index=var_names,
    columns=[f'Shock: {n}' for n in var_names]
)
print(fevd_h10.round(2).to_string())

In [21]:
fig = plot_fevd(
    fevd_results=fevd_result,
    var_names=var_names,
    title='Forecast Error Variance Decomposition',
    figsize=(14, 10)
)
plt.tight_layout()
plt.show()

In [22]:
# Save IRF
irf_records = []
for t in range(irf_result.irfs.shape[0]):
    for i, rn in enumerate(var_names):
        for j, sn in enumerate(var_names):
            irf_records.append({
                'period': t, 'response': rn, 'shock': sn,
                'var_irf': float(irf_result.irfs[t, i, j]),
                'bvar_irf': float(bvar_irf[t, i, j])
            })

irf_df = pd.DataFrame(irf_records)
irf_df.to_csv(os.path.join(output_dir, 'multivariate_irf.csv'), index=False)
print(f'Saved: {output_dir}/multivariate_irf.csv ({len(irf_df)} records)')

---
## 7. Granger Causality

In [23]:
print('=== Pairwise Granger Causality ===')
print(f'{"Causing":>15} --> {"Caused":<15} {"F-stat":>10} {"p-value":>10} {"Causal?":>10}')
print('=' * 65)

granger_matrix = np.zeros((K, K))

for i, caused in enumerate(var_names):
    for j, causing in enumerate(var_names):
        if i == j:
            continue
        gc = granger_causality(var_result, caused=caused, causing=causing)
        granger_matrix[i, j] = gc.pvalue
        marker = '*' if gc.pvalue < 0.05 else ' '
        print(f'{causing:>15} --> {caused:<15} {gc.fstat:>10.4f} '
              f'{gc.pvalue:>10.4f} {"Yes":>10} {marker}' if gc.pvalue < 0.05
              else f'{causing:>15} --> {caused:<15} {gc.fstat:>10.4f} '
              f'{gc.pvalue:>10.4f} {"No":>10} {marker}')

In [24]:
# Granger causality heatmap
fig, ax = plt.subplots(figsize=(8, 6))
gc_display = granger_matrix.copy()
np.fill_diagonal(gc_display, np.nan)

im = ax.imshow(gc_display, cmap='RdYlGn', vmin=0, vmax=0.2, aspect='auto')
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(var_names, rotation=45, ha='right')
ax.set_yticklabels(var_names)
for i in range(K):
    for j in range(K):
        if i != j:
            ax.text(j, i, f'{granger_matrix[i,j]:.3f}', ha='center', va='center',
                    fontsize=10, fontweight='bold' if granger_matrix[i,j] < 0.05 else 'normal')

plt.colorbar(im, ax=ax, label='p-value')
ax.set_title('Granger Causality p-values', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8. Structural VAR (SVAR)

Cholesky ordering: GDP -> Inflation -> Fed Funds -> Unemployment

In [25]:
svar = SVAR(var_result, method='cholesky')

print('=== SVAR (Cholesky) ===')
print(f'Ordering: {var_names}')
print(f'\nA0_inv:')
a0_df = pd.DataFrame(svar.A0_inv, index=var_names,
                     columns=[f'Shock_{n}' for n in var_names])
print(a0_df.round(4).to_string())

In [26]:
# Structural IRF
svar_irf = svar.irf(periods=20)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
periods_range = np.arange(svar_irf.irfs.shape[0])
responses = [
    (0, 2, 'GDP <- Fed Funds', 'steelblue'),
    (1, 2, 'Inflation <- Fed Funds', 'darkorange'),
    (3, 0, 'Unemployment <- GDP', 'red'),
    (2, 1, 'Fed Funds <- Inflation', 'green')
]

for ax, (resp, shock, title, color) in zip(axes.flat, responses):
    ax.plot(periods_range, svar_irf.irfs[:, resp, shock], color=color, linewidth=1.5)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.fill_between(periods_range, svar_irf.irfs[:, resp, shock], 0, alpha=0.2, color=color)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Structural Impulse Responses (SVAR Cholesky)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Spillover Index (Diebold-Yilmaz)

In [27]:
spillover = SpilloverIndex(lags=p_opt, horizon=10)
spill_result = spillover.fit(data)

print('=== Diebold-Yilmaz Spillover ===')
print(f'Total Spillover: {spill_result.total_spillover:.2f}%')

print(f'\nSpillover Table:')
spill_table = pd.DataFrame(spill_result.fevd_table, index=var_names, columns=var_names)
print(spill_table.round(2).to_string())

print(f'\nFROM others:')
for i, name in enumerate(var_names):
    print(f'  {name}: {spill_result.directional_from[i]:.2f}%')

print(f'\nTO others:')
for i, name in enumerate(var_names):
    print(f'  {name}: {spill_result.directional_to[i]:.2f}%')

print(f'\nNet spillover:')
for i, name in enumerate(var_names):
    net = spill_result.net_spillover[i]
    role = 'transmitter' if net > 0 else 'receiver'
    print(f'  {name}: {net:.2f}% ({role})')

In [28]:
# Spillover visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
im = axes[0].imshow(spill_result.fevd_table, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(K)); axes[0].set_yticks(range(K))
axes[0].set_xticklabels(var_names, rotation=45, ha='right')
axes[0].set_yticklabels(var_names)
for i in range(K):
    for j in range(K):
        axes[0].text(j, i, f'{spill_result.fevd_table[i,j]:.1f}',
                     ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title(f'Spillover Table (Total: {spill_result.total_spillover:.1f}%)', fontsize=12)

# Net spillover
net = spill_result.net_spillover
colors_net = ['green' if n > 0 else 'red' for n in net]
axes[1].barh(var_names, net, color=colors_net, alpha=0.8)
axes[1].axvline(x=0, color='black', linewidth=0.5)
axes[1].set_title('Net Spillover (+ = transmitter)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [29]:
# Save spillover
spillover_data = {
    'total_spillover': float(spill_result.total_spillover),
    'fevd_table': spill_result.fevd_table.tolist(),
    'directional_from': spill_result.directional_from.tolist(),
    'directional_to': spill_result.directional_to.tolist(),
    'net_spillover': spill_result.net_spillover.tolist(),
    'variables': var_names,
    'var_lags': p_opt,
    'horizon': 10
}

with open(os.path.join(output_dir, 'multivariate_spillover.json'), 'w') as f:
    json.dump(spillover_data, f, indent=2)

print(f'Saved: {output_dir}/multivariate_spillover.json')

---
## 10. Forecasting: VAR vs BVAR

In [30]:
# VAR forecast
h = 8
fc_var = var_result.forecast(steps=h)
fc_dates = pd.date_range(start=df.index[-1] + pd.DateOffset(months=3), periods=h, freq='QS')

print('=== VAR Forecast (8 quarters) ===')
fc_var_df = pd.DataFrame(fc_var, index=fc_dates, columns=var_names)
print(fc_var_df.round(4).to_string())

In [31]:
# BVAR forecast with posterior intervals
bvar_fc = bvar_mn_result.forecast(steps=h)

print('=== BVAR Forecast (Minnesota) ===')
fc_bvar_df = pd.DataFrame(bvar_fc['mean'], index=fc_dates, columns=var_names)
print(fc_bvar_df.round(4).to_string())

print(f'\n95% CI width (avg):')
for j, name in enumerate(var_names):
    width = np.mean(bvar_fc['upper_95'][:, j] - bvar_fc['lower_95'][:, j])
    print(f'  {name}: {width:.4f}')

In [32]:
# GRAPH: VAR vs BVAR comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
n_hist = 24

for ax, col_idx, (col, title, color) in zip(
    axes.flat, range(K), zip(var_names, titles, colors)
):
    hist_idx = df.index[-n_hist:]
    ax.plot(hist_idx, df[col].values[-n_hist:], color='black', linewidth=1.5, label='Historical')

    ax.plot(fc_dates, fc_var[:, col_idx], color=color, linewidth=2, marker='o',
            markersize=3, label='VAR')
    ax.plot(fc_dates, bvar_fc['mean'][:, col_idx], color=color, linewidth=2,
            linestyle='--', marker='s', markersize=3, label='BVAR')
    ax.fill_between(fc_dates, bvar_fc['lower_95'][:, col_idx],
                    bvar_fc['upper_95'][:, col_idx], alpha=0.15, color=color)
    ax.fill_between(fc_dates, bvar_fc['lower_68'][:, col_idx],
                    bvar_fc['upper_68'][:, col_idx], alpha=0.25, color=color)

    ax.axvline(x=hist_idx[-1], color='gray', linestyle=':', alpha=0.5)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('VAR vs BVAR Forecast', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [33]:
# Save forecasts
forecast_records = []
for t in range(h):
    for j, name in enumerate(var_names):
        forecast_records.append({
            'date': fc_dates[t].strftime('%Y-%m-%d'),
            'variable': name,
            'var_forecast': float(fc_var[t, j]),
            'bvar_mean': float(bvar_fc['mean'][t, j]),
            'bvar_median': float(bvar_fc['median'][t, j]),
            'bvar_lower_68': float(bvar_fc['lower_68'][t, j]),
            'bvar_upper_68': float(bvar_fc['upper_68'][t, j]),
            'bvar_lower_95': float(bvar_fc['lower_95'][t, j]),
            'bvar_upper_95': float(bvar_fc['upper_95'][t, j]),
        })

fc_all_df = pd.DataFrame(forecast_records)
fc_all_df.to_csv(os.path.join(output_dir, 'multivariate_forecasts.csv'), index=False)
print(f'Saved: {output_dir}/multivariate_forecasts.csv ({len(fc_all_df)} records)')

---
## 11. Report Generation

In [34]:
python_results = {
    'dataset': 'us_macro_quarterly.csv',
    'n_obs': T, 'n_vars': K,
    'var_lags': int(p_opt),
    'var_aic': float(var_result.aic),
    'var_stable': bool(var_result.is_stable),
    'coint_rank': int(coint_rank),
    'decision': decision,
    'bvar_prior': 'minnesota',
    'bvar_log_ml': float(bvar_mn_result.log_marginal_likelihood),
    'total_spillover': float(spill_result.total_spillover),
}

model_html = '<h3>Model Comparison</h3><table>'
model_html += '<tr><th>Model</th><th>Detail</th></tr>'
model_html += f'<tr><td>VAR({p_opt})</td><td>AIC={var_result.aic:.2f}, Stable={var_result.is_stable}</td></tr>'
model_html += f'<tr><td>VECM(rank={vecm_rank})</td><td>Rank={vecm_result.coint_rank}</td></tr>'
model_html += f'<tr><td>BVAR Minnesota</td><td>Log ML={bvar_mn_result.log_marginal_likelihood:.2f}</td></tr>'
model_html += f'<tr><td>BVAR Normal-Wishart</td><td>Log ML={bvar_nw_result.log_marginal_likelihood:.2f}</td></tr>'
model_html += '</table>'

tree_html = f'<h3>Decision Tree</h3><p>{decision_text}</p><ul>'
for var, order in integration_orders.items():
    tree_html += f'<li>{var}: {order}</li>'
tree_html += f'<li>Cointegration rank: {coint_rank}</li></ul>'

report_path = generate_comparison_report(
    title='Multivariate Workflow Solution: US Macro with BVAR',
    python_results=python_results,
    output_path=os.path.join(output_dir, 'multivariate_report.html'),
    extra_sections=[
        {'title': 'Decision Tree', 'html': tree_html},
        {'title': 'Models', 'html': model_html}
    ]
)
print(f'Report: {report_path}')

---
## 12. Conclusions and Recommendations

### Key Findings

1. **Integration Orders**: The unit root tests reveal the integration properties of each
   variable. The decision tree systematically selects the appropriate model.

2. **VAR vs BVAR**: The Bayesian VAR with Minnesota prior provides:
   - **Regularization** via shrinkage, preventing overfitting
   - **Uncertainty quantification** via posterior credible intervals
   - **Model comparison** via log marginal likelihood

3. **Prior Comparison**: The Minnesota prior encodes economically sensible beliefs
   (own lags more important than cross-variable lags), typically outperforming
   the Normal-Wishart prior for macro applications.

4. **Spillover**: The Diebold-Yilmaz index reveals interconnectedness structure.
   Fed funds and GDP tend to be the main shock transmitters.

5. **Structural Identification**: The Cholesky SVAR captures standard macro
   transmission mechanisms.

### Recommendations

- Use BVAR when parameters are large relative to observations
- Compare multiple priors using log marginal likelihood
- Always check VAR stability before computing IRFs
- Report posterior intervals rather than just point forecasts

---
**End of Solution Notebook**

Outputs saved:
- `multivariate_tests.json`
- `multivariate_model_selection.json`
- `multivariate_irf.csv`
- `multivariate_spillover.json`
- `multivariate_forecasts.csv`